In [ ]:
import pandas as pd
import numpy as np
import zipfile
import joblib

#Extract data
with zipfile.ZipFile('trace_models.zip') as zip_ref:
    zip_ref.extractall()

In [ ]:
def load_data():
    df = pd.read_csv('trace_train.csv', index_col='row_id')

    ids = df.index

    X = np.array(df.drop('y',axis=1))  #Models were trained without feature names(which are irelevant)
    y = df['y']

    model_A = joblib.load('model_A.joblib')
    model_B = joblib.load('model_B.joblib')
    return X, y, ids, model_A, model_B

X, y, ids, model_A, model_B = load_data()

X[:1]

array([[20000,     2,     2,     1,    24,     2,     2,    -1,    -1,
           -2,    -2,  3913,  3102,   689,     0,     0,     0,     0,
          689,     0,     0,     0,     0,   546,   144,    16,   385,
          620,    68,   982,   211,   638,   610,   767,   815,   314,
          699,   611,   857,   437,   626,   110,   612,   870,    57,
          482,   433,   123,   681,   801,   512,   436,   179]])

In [55]:
model_A, model_B

(Pipeline(steps=[('scaler', StandardScaler()),
                 ('clf',
                  MLPClassifier(alpha=1e-06, batch_size=256,
                                hidden_layer_sizes=(256, 256, 256),
                                random_state=1, verbose=True))]),
 Pipeline(steps=[('scaler', StandardScaler()),
                 ('clf',
                  MLPClassifier(alpha=1e-06, batch_size=256,
                                hidden_layer_sizes=(256, 256, 256),
                                random_state=2, verbose=True))]))

In [ ]:
#Even tho it's a very simple aproach, it works perfectly
preds_A_proba = model_A.predict_proba(X)
preds_B_proba = model_B.predict_proba(X)

preds_A_class_0 = preds_A_proba[:, 0]
preds_A_class_1 = preds_A_proba[:, 1]

preds_B_class_0 = preds_B_proba[:, 0]
preds_B_class_1 = preds_B_proba[:, 1]

def get_model(class_num, index):
    if class_num == 0:
        return 0 if preds_A_class_0[index] > preds_B_class_0[index] else 1
    else:
        return 0 if preds_A_class_1[index] > preds_B_class_1[index] else 1

answer = []
for index, pred_value in enumerate(y):
    answer.append(get_model(class_num=pred_value, index=index))

In [ ]:
#Submit
output_df = pd.DataFrame({
    'subtaskID':[1] * len(ids),
    'datapointID':ids,
    'answer':answer
})

output_df.to_csv('submission.csv', index=False)
output_df.head()

,subtaskID,datapointID,answer
0,1,0,0
1,1,1,0
2,1,2,1
3,1,3,1
4,1,4,1
